# 59 - Per-Tier BM25 Index and Retrieval

Builds a separate BM25 index for each of the four nested corpus tiers (100K, 200K, 300K, ~397K companies) and retrieves the top-1000 candidates for the 19 deep-coverage queries (5-query pilot + 14 headline queries, the only queries with a reliably deep gold standard) at each tier. Rich text is tokenized once across the full ~397K-company pool and then sliced per tier via the `in_100k`/`in_200k`/`in_300k`/`in_400k` membership flags already in `combined_pool.parquet`, rather than re-tokenizing per tier, since the four tiers are nested (each strictly contains the previous one plus more companies) and tokenization does not depend on which tier a document ends up in. BM25 itself still has to be rebuilt separately per tier, since its IDF weighting depends on document frequency across the whole indexed collection, which changes as the corpus grows. Feeds directly into notebook 60's scaling evaluation.

In [1]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

OUTPUT_DIR = Path("result/59_bm25_tiered_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIERS = [("100k", "in_100k"), ("200k", "in_200k"), ("300k", "in_300k"), ("400k", "in_400k")]
DEEP_QUERY_IDS = [1, 2, 3, 4, 5, 11, 12, 15, 34, 14, 27, 66, 72, 99, 101, 56, 82, 91, 92]
TOP_K = 1000

train_queries = json.load(open("result/08_llm_relevance_judge/train_queries.json"))
held_out_queries = json.load(open("result/08_llm_relevance_judge/held_out_queries.json"))
query_lookup = {item["query_id"]: item["query"] for item in train_queries + held_out_queries}
deep_queries = [(qid, query_lookup[qid]) for qid in DEEP_QUERY_IDS]

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")
print(f"[Load] Combined pool: {len(combined):,} companies")
for tier_name, flag_col in TIERS:
    print(f"[Load] Tier {tier_name}: {combined[flag_col].sum():,} companies")
print(f"[Load] Deep-coverage queries: {len(deep_queries)}")

[Load] Combined pool: 397,025 companies
[Load] Tier 100k: 100,000 companies
[Load] Tier 200k: 200,000 companies
[Load] Tier 300k: 300,000 companies
[Load] Tier 400k: 397,025 companies
[Load] Deep-coverage queries: 19


In [2]:
tokens_path = OUTPUT_DIR / "tokenized_corpus.json"
if tokens_path.exists():
    print("[Tokenize] Already tokenized -- loading from disk")
    tokenized_corpus = json.load(open(tokens_path))
else:
    print("[Tokenize] Tokenizing rich_text for all companies once (shared across every tier)...")
    t0 = time.time()
    tokenized_corpus = [
        word_tokenize(text.lower()) if isinstance(text, str) and text else []
        for text in combined["rich_text"].tolist()
    ]
    print(f"[Tokenize] Done in {(time.time()-t0)/60:.1f} min for {len(tokenized_corpus):,} documents")
    json.dump(tokenized_corpus, open(tokens_path, "w"))
    print(f"[Tokenize] Saved -> {tokens_path}")

[Tokenize] Tokenizing rich_text for all companies once (shared across every tier)...
[Tokenize] Done in 3.7 min for 397,025 documents
[Tokenize] Saved -> result/59_bm25_tiered_index/tokenized_corpus.json


In [3]:
all_results = []

for tier_name, flag_col in TIERS:
    out_path = OUTPUT_DIR / f"bm25_results_{tier_name}.json"
    timing_path = OUTPUT_DIR / f"bm25_timing_{tier_name}.json"
    if out_path.exists():
        print(f"[Tier {tier_name}] Already done -- loading from disk")
        tier_results = pd.read_json(out_path)
        all_results.append(tier_results)
        continue

    mask = combined[flag_col].values
    tier_domains = combined.loc[mask, "domain"].reset_index(drop=True)
    tier_corpus = [tokenized_corpus[i] for i in np.where(mask)[0]]
    print(f"[Tier {tier_name}] Building BM25 index over {len(tier_corpus):,} companies...")

    t0 = time.time()
    bm25 = BM25Okapi(tier_corpus)
    index_build_time = time.time() - t0
    print(f"[Tier {tier_name}] Index built in {index_build_time:.1f}s")

    tier_rows = []
    tier_timing_rows = []
    for qid, qtext in deep_queries:
        q_tokens = word_tokenize(qtext.lower())
        t0 = time.perf_counter()
        scores = bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:TOP_K]
        query_ms = (time.perf_counter() - t0) * 1000
        for rank, idx in enumerate(top_idx):
            tier_rows.append({
                "tier": tier_name, "query_id": qid, "rank": rank + 1,
                "domain": tier_domains.iloc[idx], "bm25_score": float(scores[idx]),
                "query_latency_ms": query_ms,
            })
        tier_timing_rows.append({
            "tier": tier_name, "query_id": qid, "n_companies": len(tier_corpus),
            "index_build_s": index_build_time, "query_latency_ms": query_ms,
        })

    # Save results AND this tier's timing together -- if the job times out on a later tier,
    # this tier's timing must not depend on the notebook reaching the final summary cell.
    tier_results = pd.DataFrame(tier_rows)
    tier_results.to_json(out_path, orient="records", indent=2)
    pd.DataFrame(tier_timing_rows).to_json(timing_path, orient="records", indent=2)
    print(f"[Tier {tier_name}] Saved -> {out_path}")
    all_results.append(tier_results)

bm25_all = pd.concat(all_results, ignore_index=True)
bm25_all.to_json(OUTPUT_DIR / "bm25_results_all_tiers.json", orient="records", indent=2)
print(f"[Done] Combined results: {len(bm25_all):,} rows across {bm25_all['tier'].nunique()} tiers")

[Tier 100k] Building BM25 index over 100,000 companies...
[Tier 100k] Index built in 3.4s
[Tier 100k] Saved -> result/59_bm25_tiered_index/bm25_results_100k.json
[Tier 200k] Building BM25 index over 200,000 companies...
[Tier 200k] Index built in 8.3s
[Tier 200k] Saved -> result/59_bm25_tiered_index/bm25_results_200k.json
[Tier 300k] Building BM25 index over 300,000 companies...
[Tier 300k] Index built in 12.9s
[Tier 300k] Saved -> result/59_bm25_tiered_index/bm25_results_300k.json
[Tier 400k] Building BM25 index over 397,025 companies...
[Tier 400k] Index built in 15.0s
[Tier 400k] Saved -> result/59_bm25_tiered_index/bm25_results_400k.json
[Done] Combined results: 76,000 rows across 4 tiers


In [4]:
timing_files = sorted(OUTPUT_DIR.glob("bm25_timing_*.json"))
timing_df = pd.concat([pd.read_json(f) for f in timing_files], ignore_index=True) if timing_files else pd.DataFrame()
if len(timing_df):
    summary = timing_df.groupby("tier").agg(n_companies=("n_companies", "first"), index_build_s=("index_build_s", "first"), avg_query_latency_ms=("query_latency_ms", "mean")).reindex(["100k", "200k", "300k", "400k"])
    summary.to_csv(OUTPUT_DIR / "bm25_timing_summary.csv")
    print(summary.to_string())
else:
    print("No timing files found yet -- run the previous cell first.")

      n_companies  index_build_s  avg_query_latency_ms
tier                                                  
100k       100000       3.444433            172.080309
200k       200000       8.309424            349.914344
300k       300000      12.925721            501.590312
400k       397025      14.999213            647.086609
